# [실습] 군집 — 고객을 묶고 의심하기

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# Chapter 6 실습. 고객을 몇 개 그룹으로 나눠야 할까?

**필수 머신러닝 · Chapter 6**

| Part | 주제 |
| --- | --- |
| Part 1 | 고객을 숫자로: 정제와 RFM |
| Part 2 | K-means로 묶기 |
| Part 3 | 묶고 의심하기 |
| Part 4 | CRM팀에 보고하기 |

## 분석 의뢰서

> **의뢰 부서**: 온라인 도매몰 CRM팀
> **목적**: 고객 4,000여 명을 몇 개 그룹으로 나눠 그룹별로 다른 관리 정책(할인, 리마인드, 전담 상담)을 적용하고 싶다.
> **제약**: 그룹은 운영팀이 기억하고 관리할 수 있을 만큼 적어야 한다(5개 이내). 그룹이 바뀌면 정책도 바꿔야 하므로 자주 바뀌면 곤란하다.
> **질문**: 고객을 몇 개 그룹으로 나눠야 하고, 각 그룹은 어떤 고객이며, 그 그룹은 믿을 만한가?

## 이 실습의 질문

> **CRM팀에 줄 고객 그룹은 몇 개이고, 각 그룹의 기준은 무엇인가? 그 그룹이 "진짜"라고 CRM팀을 설득하는 근거는 무엇이고, 어디까지가 분석가의 선택인가?**

Chapter 1~5와 달리 이번에는 맞힐 정답이 없습니다. 그래서 "모델의 점수" 대신 **선택을 바꾸면 결과가 얼마나 바뀌는가**와 **군집에 쓰지 않은 데이터로 확인하기**에 시간을 씁니다.

## 이 노트북을 쓰는 법

| 표시 | 뜻 |
| --- | --- |
| **[직접 작성]** | 빈 코드 셀을 직접 채웁니다. 강의 교안의 코드를 보지 않고 먼저 써 보세요 |
| **[판단]** | 코드가 아니라 생각을 적는 칸입니다. 셀을 더블클릭해 `나의 답:` 뒤에 적으세요 |
| **[제공]** | 미리 채워 둔 코드입니다. 읽고 실행하세요 |
| **힌트** | 막혔을 때만 펼치세요 |
| **먼저 예상하기** | 결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다 |

## 준비

UCI Online Retail II 데이터를 CSV로 바꾼 `online_retail_II.csv`(약 97MB, 1,067,371행)를 이 노트북과 같은 폴더에 두세요. 파일이 크므로 Colab에서는 Google Drive에 올려 두고 연결해서 읽는 것을 권합니다.

In [1]:
# [제공] 이 실습에서 쓰는 라이브러리
import platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score

# 그래프의 한글 글꼴 (환경에 맞는 글꼴이 없으면 글자가 깨질 수 있습니다)
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

In [2]:
# [환경] 그래프 한글 글꼴 등록 (같은 폴더의 fonts/NanumGothic-Regular.ttf)
import os
import matplotlib.font_manager as fm
for _p in ["fonts/NanumGothic-Regular.ttf", "../fonts/NanumGothic-Regular.ttf", "NanumGothic-Regular.ttf"]:
    if os.path.exists(_p):
        fm.fontManager.addfont(_p)
        plt.rcParams["font.family"] = fm.FontProperties(fname=_p).get_name()
        break
plt.rcParams["axes.unicode_minus"] = False

---
# Part 1. 고객을 숫자로: 정제와 RFM

## 1-1. 데이터 불러오기

**[제공]** 송장 번호와 상품 코드는 글자로, 주문 일시는 날짜로 읽습니다. 행이 100만 개가 넘어 읽는 데 수십 초가 걸릴 수 있습니다.

In [3]:
# [제공] 데이터 불러오기
raw = pd.read_csv("online_retail_II.csv", dtype={"Invoice": str, "StockCode": str}, parse_dates=["InvoiceDate"])
print(raw.shape)
raw.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 1-2. 정제: 실제로 산 기록만 남기기

**[직접 작성]** 다음 네 가지의 비율을 각각 출력한 뒤, 네 가지를 모두 뺀 데이터를 `sales`에 저장하세요. `sales`의 행 수와 고객 수(`CustomerID`의 고유값 수)도 출력합니다.

| 정제 항목 | 조건 |
| --- | --- |
| 고객 ID 없음 | `CustomerID`가 빈 값 |
| 취소 송장 | `Invoice`가 `C`로 시작 |
| 수량 0 이하 | `Quantity <= 0` |
| 가격 0 이하 | `Price <= 0` |

그다음 `sales`에 구매 금액 열 `amount`(= `Quantity` × `Price`)를 만드세요.

<details><summary>힌트</summary>

취소 송장은 `raw["Invoice"].str.startswith("C")`, 빈 값은 `raw["CustomerID"].isna()`. 비율은 조건에 `.mean()`을 붙입니다.
</details>

In [4]:
n = len(raw)
print(f"고객 ID 없음:       {raw['CustomerID'].isna().mean() * 100:6.2f}%  ({raw['CustomerID'].isna().sum():,}행)")
print(f"취소 송장(C 시작):  {raw['Invoice'].str.startswith('C').mean() * 100:6.2f}%  ({raw['Invoice'].str.startswith('C').sum():,}행)")
print(f"수량 0 이하:        {(raw['Quantity'] <= 0).mean() * 100:6.2f}%  ({(raw['Quantity'] <= 0).sum():,}행)")
print(f"가격 0 이하:        {(raw['Price'] <= 0).mean() * 100:6.2f}%  ({(raw['Price'] <= 0).sum():,}행)")

sales = raw[raw["CustomerID"].notna() & ~raw["Invoice"].str.startswith("C")
            & (raw["Quantity"] > 0) & (raw["Price"] > 0)].copy()
sales["CustomerID"] = sales["CustomerID"].astype(int)
sales["amount"] = sales["Quantity"] * sales["Price"]
print()
print(f"sales: {len(sales):,}행 | 고객 {sales['CustomerID'].nunique():,}명 | 전체 {n:,}행의 {len(sales) / n * 100:.1f}%")

고객 ID 없음:        22.77%  (243,007행)


취소 송장(C 시작):    1.83%  (19,494행)
수량 0 이하:          2.15%  (22,950행)
가격 0 이하:          0.58%  (6,207행)



sales: 805,549행 | 고객 5,878명 | 전체 1,067,371행의 75.5%


> **[판단]** 반품을 빼지 않고 구매액에서 빼는 방식으로 정제했다면 군집 결과에 어떤 차이가 생길 수 있을까요? 이 정제 기준을 보고서에 적어야 하는 이유는 무엇인가요?
>
> 나의 답:

반품(음수 수량·취소 송장)을 빼지 않고 **구매액에서 빼는** 방식으로 정제했다면, 금액(M)이 상쇄되어 ⑴ 많이 사고 많이 반품한 고객의 M이 작아져 **핵심 고객이 중간 그룹으로 내려가고**, ⑵ 반품이 잦은 고객이 "저가 고객"으로 분류되며, ⑶ F(송장 수)에는 반품 송장이 섞여 재구매 빈도가 과대 계산됩니다. 반품은 "구매가 없었던 일"이 아니라 **서비스 실패의 흔적**이므로, 빼는 대신 별도 지표(반품률)로 남겨 두는 것이 옳습니다. 이 정제 기준을 보고서에 적어야 하는 이유는, 그룹의 경계가 **분석가가 정한 규칙**(무엇을 구매로 볼 것인가)에서 나오기 때문입니다. 기준을 바꾸면 그룹 구성이 달라지는데, 보고서에 기준이 없으면 CRM팀은 그룹이 데이터의 성질이라고 오해합니다.


## 1-3. 기간 나누기와 RFM 계산

| 구간 | 기간 | 용도 |
| --- | --- | --- |
| 군집용 | 2009-12-01 ~ 2010-11-30 | 고객을 묶는다 |
| 검증용 | 2010-12-01 ~ 2011-11-30 | 묶은 그룹이 다음 해에 실제로 다르게 행동했는지 확인한다 |

**[직접 작성]** `sales`를 두 기간으로 나눠 `period1`, `period2`에 저장하세요. 그다음 **군집용 기간(`period1`)으로만** 고객마다 RFM을 계산해 `rfm`(고객 ID가 인덱스, 열 `R`, `F`, `M`)에 저장하고, 고객 수를 출력하세요. 기준일은 2010-12-01입니다.

| 지표 | 계산 |
| --- | --- |
| `R` | 기준일 − 마지막 구매일 (일) |
| `F` | 송장 수 (`Invoice`의 고유값 수) |
| `M` | 구매 금액의 합 (`amount`의 합) |

<details><summary>힌트</summary>

```python
ref = pd.Timestamp("2010-12-01")
rfm = period1.groupby("CustomerID").agg(
    R=("InvoiceDate", lambda s: (ref - s.max()).days),
    F=("Invoice", "nunique"),
    M=("amount", "sum"),
)
```
</details>

In [5]:
period1 = sales[(sales["InvoiceDate"] >= "2009-12-01") & (sales["InvoiceDate"] < "2010-12-01")]
period2 = sales[(sales["InvoiceDate"] >= "2010-12-01") & (sales["InvoiceDate"] < "2011-12-01")]
print(f"군집용 {len(period1):,}행 | 검증용 {len(period2):,}행")

snapshot = pd.Timestamp("2010-12-01")
g1 = period1.groupby("CustomerID")
rfm = pd.DataFrame({
    "R": (snapshot - g1["InvoiceDate"].max()).dt.days,
    "F": g1["Invoice"].nunique(),
    "M": g1["amount"].sum(),
})
print(f"군집 대상 고객: {len(rfm):,}명")
rfm.head()

군집용 393,160행 | 검증용 395,084행
군집 대상 고객: 4,266명


,R,F,M
CustomerID,,,
12346,155,11,372.86
12347,30,1,611.53
12348,64,1,222.16
12349,33,3,2671.14
12351,1,1,300.93


**[직접 작성]** `rfm`의 분포를 확인하세요. 최솟값, 중앙값, 평균, 상위 10%, 상위 1%, 최댓값, 표준편차가 보이도록 출력합니다.

<details><summary>힌트</summary>

`rfm.describe(percentiles=[0.5, 0.9, 0.99]).round(1)`
</details>

In [6]:
rfm.describe(percentiles=[0.5, 0.9, 0.99]).round(1)

,R,F,M
count,4266.0,4266.0,4266.0
mean,87.2,4.3,1997.2
std,95.1,7.8,8601.8
min,0.0,1.0,3.0
50%,47.0,2.0,689.1
90%,247.0,9.0,3683.4
99%,358.0,29.4,19518.9
max,364.0,197.0,321329.7


> **[판단]** F와 M의 평균과 중앙값은 얼마나 다른가요? 이 차이는 분포가 어떤 모양이라는 뜻인가요? 거리로 고객을 묶을 때 이 모양이 어떤 문제를 만들 것 같은가요?
>
> 나의 답:

F와 M 모두 **평균이 중앙값보다 훨씬 큽니다**. F는 중앙값 2회, 평균 4.3회(2.2배), M은 중앙값 689, 평균 1,997(2.9배)이고, 상위 1%가 19,519, 최댓값이 321,330입니다. 이는 분포가 **오른쪽으로 길게 꼬인 모양**(소수 고액 고객이 평균을 끌어올림)이라는 뜻입니다. 거리로 고객을 묶을 때 이 모양은 문제를 만듭니다. 유클리드 거리에서는 **단위가 크고 꼬리가 긴 M이 거의 모든 거리를 지배**해서, K-means가 사실상 "M 상위 몇 명 vs 나머지"로만 나뉘게 됩니다. 실제로 원본 그대로 돌린 결과가 [4137, 3, 13, 113]으로, 소수 고액 고객만 떨어져 나오고 나머지 96%가 한 덩어리가 되었습니다.


---
# Part 2. K-means로 묶기

## 2-1. 손으로 돌려 보기

교안 2-1의 숫자 6개(1, 2, 3, 10, 11, 12)를 $k = 2$로 나눠 봅니다.

**[직접 작성]** `KMeans`에 초기 중심을 1과 2로 직접 주고(`init=np.array([[1], [2]])`, `n_init=1`) 학습한 뒤, 최종 중심, 그룹, inertia를 출력하세요. 이어서 같은 6개 점에 대해 `silhouette_score`로 실루엣 점수를 계산하세요.

In [7]:
points = np.array([[1], [2], [3], [10], [11], [12]], dtype=float)
km_manual = KMeans(n_clusters=2, init=np.array([[1.0], [2.0]]), n_init=1, random_state=42).fit(points)
print("최종 중심:", np.sort(km_manual.cluster_centers_.ravel()))
print("그룹:", km_manual.labels_)
print("inertia:", km_manual.inertia_)
print("실루엣:", round(silhouette_score(points, km_manual.labels_), 3))

최종 중심: [ 2. 11.]
그룹: [0 0 0 1 1 1]
inertia: 4.0
실루엣: 0.85


/workspace/.local/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py:134: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/workspace/.local/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 272, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


> **[판단]** 나쁜 초기값(1과 2)에서 시작했는데도 최종 결과는 눈으로 봐도 자연스러운 두 덩어리인가요? 교안 2-1의 손 계산(중심 2와 11, inertia 4.0)과 같은가요?
>
> 나의 답:

나쁜 초기값(중심 1과 2)에서 시작했는데도 최종 중심은 **2와 11**, 그룹은 [0,0,0,1,1,1], **inertia 4.0**으로, 교안 2-1의 손 계산과 **완전히 같습니다**. 실루엣도 0.85로 높습니다. 이유는 K-means가 (1) 각 점을 가장 가까운 중심에 배정하고 (2) 중심을 평균으로 옮기는 일을 반복하면서 **스스로 좋은 해로 이동**하기 때문입니다. 1과 2에 몰려 있던 중심 중 하나가 평균을 따라 11 쪽으로 이동했습니다. 다만 이 예는 "운이 좋았던" 경우로, 여러 번 시작(n_init=10)하는 이유는 나쁜 지역해에 갇히는 경우가 있기 때문입니다.


## 2-2. 전처리가 군집을 정한다

### 먼저 예상하기

결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다. 다음 절에서 같은 K-means($k = 4$)를 세 가지 전처리에 적용합니다.

> **[판단]** 예상: RFM 원본 그대로 K-means를 돌리면 네 그룹의 크기는 어떻게 나올까요? (예: 비슷한 크기 네 개 / 하나는 아주 크고 나머지는 작음)
>
> 나의 답:

예상: 원본 그대로 돌리면 **하나가 아주 크고 나머지는 아주 작은** 모양이 될 것 같습니다. M의 단위가 수십만까지 커서 거의 모든 거리를 M이 결정하고, K-means는 "M이 큰 소수"와 "나머지 전부"로 나뉠 것이기 때문입니다.


**[제공]** K-means를 돌려 그룹 크기와 실루엣을 보여 주는 함수입니다. `n_init=10`(여러 번 시작해 가장 좋은 결과), `random_state=42`로 고정합니다.

In [8]:
# [제공] K-means를 돌리고 그룹 크기와 실루엣을 출력
def run_kmeans(X, k, seed=42, verbose=True):
    km = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(X)
    if verbose:
        print(f"k={k} | 그룹 크기 {np.bincount(km.labels_)} | inertia {km.inertia_:,.0f} | 실루엣 {silhouette_score(X, km.labels_):.3f}")
    return km

**[직접 작성]** 세 가지 전처리로 행렬을 만들고 각각 `run_kmeans(X, 4)`를 실행하세요.

1. `X_raw`: R, F, M 원본 그대로
2. `X_std`: 세 열을 `StandardScaler`로 표준화
3. `X_log`: F와 M에 `np.log1p`를 적용한 뒤(R은 그대로) 세 열을 표준화

<details><summary>힌트</summary>

```python
X_log = StandardScaler().fit_transform(np.column_stack([rfm["R"], np.log1p(rfm["F"]), np.log1p(rfm["M"])]))
```
</details>

In [9]:
X_raw = rfm[["R", "F", "M"]].values
X_std = StandardScaler().fit_transform(X_raw)
X_log = StandardScaler().fit_transform(np.column_stack([rfm["R"], np.log1p(rfm["F"]), np.log1p(rfm["M"])]))

print("[원본 그대로]"); km_raw = run_kmeans(X_raw, 4)
print("[표준화]");     km_std = run_kmeans(X_std, 4)
print("[로그+표준화]"); km_log = run_kmeans(X_log, 4)

[원본 그대로]


k=4 | 그룹 크기 [4137    3   13  113] | inertia 34,806,475,315 | 실루엣 0.885
[표준화]


k=4 | 그룹 크기 [3160 1049   52    5] | inertia 3,436 | 실루엣 0.607
[로그+표준화]


k=4 | 그룹 크기 [1290  613  912 1451] | inertia 3,304 | 실루엣 0.373


> **[판단]** 예상은 맞았나요? 실루엣이 가장 높은 전처리는 무엇이고, 그 결과를 CRM팀에 줄 수 있나요? 실루엣이 높은데도 쓸 수 없다면 그 이유는 무엇인가요?
>
> 나의 답:

예상이 맞았습니다. 원본 그대로는 **[4137, 3, 13, 113]** 으로 96.6%가 한 그룹에 몰렸습니다. 표준화하면 [3160, 1049, 52, 5], 로그+표준화하면 [1290, 613, 912, 1451]로 비교적 고르게 나뉩니다. 그런데 **실루엣이 가장 높은 것은 원본 그대로(0.885)** 이고, 로그+표준화가 0.373으로 가장 낮습니다. 실루엣이 높아도 쓸 수 없습니다. 원본의 높은 실루엣은 "소수 고액 고객만 아주 멀리 떨어져 있어서" 생긴 것이고, 나머지 4,137명은 **한 덩어리로 뭉개져** CRM팀이 쓸 수 있는 4개 그룹이 아닙니다. 실루엣은 "얼마나 잘 분리됐나"만 재고, "그룹이 쓸모 있는가"는 재지 않습니다.


## 2-3. $k$ 고르기

**[직접 작성]** 로그 + 표준화한 `X_log`로 $k$를 2부터 8까지 바꿔 가며 `run_kmeans`를 실행하세요. inertia가 줄어드는 양(직전 $k$ 대비)도 계산해 보세요.

In [10]:
prev_inertia = None
rows = []
for k in range(2, 9):
    km_k = run_kmeans(X_log, k, verbose=False)
    sil = silhouette_score(X_log, km_k.labels_)
    drop = None if prev_inertia is None else prev_inertia - km_k.inertia_
    rows.append({"k": k, "그룹 크기": np.bincount(km_k.labels_).tolist(),
                 "inertia": round(km_k.inertia_, 1), "줄어든 양": None if drop is None else round(drop, 1),
                 "실루엣": round(sil, 3)})
    prev_inertia = km_k.inertia_
k_table = pd.DataFrame(rows)
k_table

,k,그룹 크기,inertia,줄어든 양,실루엣
0,2,"[1852, 2414]",6903.7,NaN,0.397
1,3,"[1962, 1334, 970]",4351.7,2551.9,0.409
2,4,"[1290, 613, 912, 1451]",3303.9,1047.8,0.373
3,5,"[880, 1243, 262, 959, 922]",2838.1,465.8,0.340
4,6,"[418, 904, 926, 641, 247, 1130]",2430.3,407.9,0.340
5,7,"[871, 880, 415, 248, 365, 1095, 392]",2171.8,258.5,0.333
6,8,"[888, 421, 527, 687, 357, 131, 394, 861]",1948.4,223.4,0.310


> **[판단]** 엘보(inertia가 크게 줄다 완만해지는 지점)와 실루엣이 허용하는 후보는 무엇인가요? 숫자만으로 최종 $k$를 정할 수 있나요? 의뢰서의 제약(5개 이내, 그룹별로 다른 정책)을 넣으면 무엇을 고르겠습니까?
>
> 나의 답:

엘보는 **k=3**(inertia 감소 2,552)과 **k=4**(1,048) 사이에서 꺾이고, k=5부터는 감소가 466·408·259·223으로 급격히 줄어듭니다. 실루엣은 k=2(0.397)·k=3(0.409)이 가장 높고 k=4는 0.373으로 조금 낮습니다. 숫자만으로는 **k=3 또는 k=4**가 후보이고, k=3을 고를 근거(실루엣 최고)도 있습니다. 그러나 최종 k는 숫자가 아니라 **의뢰서의 제약**이 정합니다. CRM팀은 "5개 이내"를 요구했고, "그룹별로 다른 정책"을 적용하려면 그룹이 서로 다른 행동을 보여야 합니다. 3-2에서 k=4의 네 그룹은 다음 해 재구매율이 34.8%~95.4%로 갈렸습니다. k=3은 "성장 중 + 최근 1회 구매"를 한 덩어리로 묶어, 재구매율 52%와 79%가 섞입니다. 정책을 나누려면 **k=4**가 낫습니다. 즉 숫자는 후보를 좁히고, **쓸모**가 최종 결정을 합니다.


## 2-4. 프로필과 이름 붙이기

**[직접 작성]** $k = 4$로 최종 모델을 만들어 그룹 번호를 `rfm["cluster"]`에 저장하고, 그룹별 고객 수와 R, F, M의 **중앙값**을 표로 출력하세요.

In [11]:
km4 = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_log)
rfm["cluster"] = km4.labels_
profile = rfm.groupby("cluster").agg(고객수=("R", "size"), R=("R", "median"), F=("F", "median"), M=("M", "median"))
profile["비중"] = (profile["고객수"] / len(rfm) * 100).round(1)
profile

,고객수,R,F,M,비중
cluster,,,,,
0,1290,43.0,1.0,328.860,30.2
1,613,12.0,10.0,4214.770,14.4
2,912,243.0,1.0,286.275,21.4
3,1451,34.0,4.0,1307.400,34.0


**[직접 작성]** 위 프로필 표를 보고 그룹마다 이름을 붙여 `names` 딕셔너리에 저장하세요. 그룹 번호는 실행할 때마다 다를 수 있으니, **반드시 위 표를 보고** 번호와 이름을 맞춥니다. 이 이름은 Part 3에서 표를 읽기 쉽게 하는 데 씁니다.

예: `names = {0: "최근 1회 구매", 1: "핵심 고객", 2: "휴면 1회 구매", 3: "성장 중"}`

In [12]:
names = {0: "최근 1회 구매", 1: "핵심 고객", 2: "휴면 1회 구매", 3: "성장 중"}
print(names)

{0: '최근 1회 구매', 1: '핵심 고객', 2: '휴면 1회 구매', 3: '성장 중'}


**[제공]** 이름을 붙인 그룹 열 `group`을 만듭니다. (위 셀에서 이미 만들었다면 다시 실행해도 같습니다.)

In [13]:
# [제공] 이름 붙인 그룹 열
rfm["group"] = rfm["cluster"].map(names)
rfm["group"].value_counts()

group
성장 중        1451
최근 1회 구매    1290
휴면 1회 구매     912
핵심 고객        613
Name: count, dtype: int64

> **[판단]** 이름은 어떤 숫자를 근거로 붙였나요? 같은 표를 보고 다른 분석가는 다른 이름을 붙일 수 있을까요? 이름을 보고서에 쓸 때 함께 적어야 할 것은 무엇인가요?
>
> 나의 답:

이름은 각 그룹의 **R·F·M 중앙값**을 근거로 붙였습니다. ① 핵심 고객(613명): R 12일·F 10회·M 4,215 — 최근이고 자주, 많이 산다. ② 성장 중(1,451명): R 34일·F 4회·M 1,307 — 최근이며 구매가 늘어나는 중. ③ 최근 1회 구매(1,290명): R 43일·F 1회·M 329 — 최근이지만 한 번만 샀다. ④ 휴면 1회 구매(912명): R 243일·F 1회·M 286 — 오래전에 한 번 샀다.
같은 표를 보고 다른 분석가는 다른 이름을 붙일 수 있습니다. 예를 들어 ③을 "신규 체험 고객", ②를 "충성 고객 후보"라고 부를 수 있습니다. 이름은 **데이터가 아니라 해석**입니다. 그래서 보고서에는 이름만 쓰지 말고, **그 이름을 붙인 근거 숫자(R·F·M 중앙값)와 고객 수**를 함께 적어야 합니다. 그래야 CRM팀이 "핵심 고객"의 뜻을 우리와 같게 이해합니다.


---
# Part 3. 묶고 의심하기

## 3-1. 선택을 바꾸면 그룹이 바뀌는가

### 먼저 예상하기

결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다.

> **[판단]** 예상: $k = 3$의 그룹들은 $k = 4$에서 "하나가 둘로 쪼개지는" 모양일까요, 아니면 더 복잡하게 섞일까요? 그리고 초기값만 바꿔 다시 돌리면 결과가 달라질까요?
>
> 나의 답:

예상: k=3의 그룹들은 k=4에서 **하나가 둘로 쪼개지는** 모양일 것 같습니다. 특히 "성장 중"과 "최근 1회 구매"는 둘 다 최근 구매 고객이라 k=3에서 한 덩어리일 것으로 봤습니다. 초기값을 바꿔 다시 돌리면 결과는 **거의 같을 것**(완전히 같은 그룹이 있으면 이동이 없으므로)이라고 예상했습니다.


**[직접 작성]** 세 가지를 확인하세요.

1. $k = 3$으로 돌린 결과를 `rfm["k3"]`에 저장하고, `pd.crosstab(rfm["k3"], rfm["group"])`으로 $k = 4$의 그룹과 겹쳐 보기 ($k = 3$ 그룹의 R, F, M 중앙값도 함께 출력)
2. $k = 4$를 `random_state` 0~4로 다섯 번 다시 돌려, 최종 모델(`km4`)과의 ARI(`adjusted_rand_score`)
3. 계층적 군집(`AgglomerativeClustering(n_clusters=4, linkage="ward")`)과 최종 모델의 ARI

<details><summary>힌트</summary>

`run_kmeans(X_log, 4, seed=s, verbose=False).labels_`로 다시 돌린 결과를 얻고, `adjusted_rand_score(km4.labels_, 그 결과)`로 비교합니다.
</details>

In [14]:
rfm["k3"] = run_kmeans(X_log, 3, verbose=False).labels_
print("[k=3 그룹별 R·F·M 중앙값]")
print(rfm.groupby("k3")[["R", "F", "M"]].median().assign(고객수=rfm.groupby("k3").size()))
print()
print("[k=3 × k=4 교차표]")
print(pd.crosstab(rfm["k3"], rfm["group"]))

print()
print("[초기값(random_state)에 따른 ARI]")
for s in range(5):
    lab = run_kmeans(X_log, 4, seed=s, verbose=False).labels_
    print(f"  seed={s}: ARI {adjusted_rand_score(km4.labels_, lab):.3f}")

agg = AgglomerativeClustering(n_clusters=4, linkage="ward").fit(X_log)
print()
print("[계층적 군집(ward) vs K-means(seed=42)] ARI:", round(adjusted_rand_score(km4.labels_, agg.labels_), 3))

[k=3 그룹별 R·F·M 중앙값]
        R    F        M   고객수
k3                           
0    40.0  2.0   510.10  1962
1    19.0  7.0  2486.95  1334
2   235.0  1.0   280.10   970

[k=3 × k=4 교차표]
group  성장 중  최근 1회 구매  핵심 고객  휴면 1회 구매
k3                                    
0       716      1242      0         4
1       721         0    613         0
2        14        48      0       908

[초기값(random_state)에 따른 ARI]
  seed=0: ARI 1.000
  seed=1: ARI 0.997
  seed=2: ARI 1.000
  seed=3: ARI 0.998
  seed=4: ARI 0.999



[계층적 군집(ward) vs K-means(seed=42)] ARI: 0.81


> **[판단]** 예상은 맞았나요? $k = 3$의 한 그룹이 $k = 4$에서 어떻게 나뉘고 합쳐지는지 숫자로 설명하세요. 초기값에 대한 ARI와 계층적 군집과의 ARI를 보고, 이 군집이 "안정적"인지와 "유일한 정답"인지를 구분해 말해 보세요.
>
> 나의 답:

예상이 맞았습니다. k=3의 그룹0(1,962명)은 k=4에서 **성장 중 716명 + 최근 1회 구매 1,242명**으로 쪼개졌습니다(휴면 4명 섞임). k=3의 그룹1(1,334명)은 **성장 중 721 + 핵심 고객 613**으로 쪼개졌고, 그룹2(970명)는 **휴면 908 + 최근 48 + 성장 14**로 거의 그대로입니다. 즉 k=3은 "최근에 산 사람"을 하나로 묶었고, k=4는 그 안을 **구매 횟수·금액(F·M)** 으로 다시 나눕니다.
초기값에 대한 ARI는 seed 0~4에서 **0.997~1.000**으로, 이 군집은 **안정적**입니다(같은 데이터·같은 k에서 시작점을 바꿔도 같은 답). 그러나 계층적 군집(ward)과의 ARI는 **0.81**로, 19%가 다르게 묶입니다. 즉 "안정적"이지만 **"유일한 정답"은 아닙니다**. 방법을 바꾸면 다른 그룹이 나오므로, 이 그룹은 데이터가 품은 하나의 정답이 아니라 **우리가 고른 방법의 결과**입니다.


## 3-2. 다음 해에 정말 달랐는가

정답이 없으니 "군집이 맞는가"는 물을 수 없습니다. 대신 "**군집이 앞으로의 행동을 가르는가**"를 봅니다. 검증용 기간(`period2`)은 군집에 한 번도 쓰지 않았습니다.

**[직접 작성]** 고객마다 검증용 기간의 구매 금액 합을 `rfm["next_amount"]`(구매가 없으면 0), 재구매 여부를 `rfm["repurchase"]`(구매 금액 > 0)에 저장하고, 그룹별로 고객 수, 다음 해 재구매율, 구매액 평균, 구매액 중앙값을 출력하세요. 전체 재구매율도 출력합니다.

<details><summary>힌트</summary>

`period2.groupby("CustomerID")["amount"].sum()`을 `.reindex(rfm.index).fillna(0)`으로 군집 대상 고객에 맞춥니다.
</details>

In [15]:
next_amount = period2.groupby("CustomerID")["amount"].sum().reindex(rfm.index).fillna(0)
rfm["next_amount"] = next_amount
rfm["repurchase"] = (rfm["next_amount"] > 0).astype(int)

table = rfm.groupby("group").agg(고객수=("next_amount", "size"), 다음해_재구매율=("repurchase", "mean"),
                                 구매액_평균=("next_amount", "mean"), 구매액_중앙값=("next_amount", "median"))
table["다음해_재구매율"] = table["다음해_재구매율"].round(3)
table["구매액_평균"] = table["구매액_평균"].round(1)
table["구매액_중앙값"] = table["구매액_중앙값"].round(1)
print("전체 재구매율:", round(rfm["repurchase"].mean(), 3), "| 전체 구매액 평균:", round(rfm["next_amount"].mean(), 1))
table

전체 재구매율: 0.636 | 전체 구매액 평균: 1685.3


,고객수,다음해_재구매율,구매액_평균,구매액_중앙값
group,,,,
성장 중,1451,0.785,1280.9,668.1
최근 1회 구매,1290,0.521,379.0,89.7
핵심 고객,613,0.954,7544.2,3257.4
휴면 1회 구매,912,0.348,238.5,0.0


> **[판단]** 다음 해 재구매율은 그룹마다 얼마나 다른가요? 특히 "휴면 1회 구매"와 "최근 1회 구매"는 F와 M이 거의 같은데 재구매율은 어떤가요? 이 결과는 $k = 4$를 고른 선택에 대해 무엇을 말해 주나요?
>
> 나의 답:

다음 해 재구매율은 그룹마다 크게 다릅니다. **핵심 고객 95.4%**(구매액 평균 7,544), **성장 중 78.5%**(1,281), **최근 1회 구매 52.1%**(379), **휴면 1회 구매 34.8%**(239)이고 전체는 63.6%입니다. 특히 "휴면 1회 구매"와 "최근 1회 구매"는 F(1회)·M(약 300)이 거의 같은데 재구매율은 **34.8% vs 52.1%**로 17%p 차이가 납니다. 두 그룹을 가른 것은 **R(최근성)** 하나였고, 그 R이 다음 해 행동을 실제로 갈랐습니다. 이는 k=4를 고른 선택(특히 R로 1회 구매 고객을 둘로 나눈 것)이 **쓸모 있었다**는 근거입니다.


> **[판단]** 이 검증 결과가 말하지 **않는** 것을 두 가지 적어 보세요. (예: 그룹이 유일한가? 그룹이 원인인가? 정책이 효과가 있는가?)
>
> 나의 답:

이 검증이 말하지 **않는** 것:
① **그룹이 유일한가** — 계층적 군집과의 ARI가 0.81이므로 다른 방법은 다른 그룹을 냅니다. 4개가 자연의 정답이 아닙니다.
② **그룹이 원인인가** — "휴면이라서 재구매하지 않는다"가 아니라, 휴면 고객에게는 원래 재구매 성향이 낮은 다른 성질(예: 1회성 선물 구매)이 있을 수 있습니다.
③ **정책이 효과가 있는가** — 이 검증은 그룹 간 차이를 본 것이지, 그룹별 정책(할인·리마인드)을 시행했을 때 행동이 바뀌는지는 전혀 보여 주지 않습니다. 정책 효과는 무작위 배정 실험으로만 알 수 있습니다.
④ 다음 해 재구매율 차이가 **R·F·M만의 효과인지**, 다른 변수(국가, 상품 종류) 때문인지도 알 수 없습니다.


## 3-3. 사람의 규칙과 비교하기

CRM 현장에서는 모델 없이 **RFM 점수**로 고객을 나눠 왔습니다. R, F, M을 고객 수가 같은 다섯 구간으로 나눠 1~5점을 주고(R은 최근일수록 5점), 점수 조합으로 세그먼트를 정합니다.

| 규칙 세그먼트 | 조건 |
| --- | --- |
| VIP | R 4~5점이고 F 4~5점 |
| 최근 신규 | R 4~5점 (F는 낮음) |
| 이탈 위험 우량 | R 1~2점이고 F 4~5점 |
| 휴면 | R 1~2점 (F는 낮음) |
| 일반 | 나머지 |

**[제공]** RFM 점수와 규칙 세그먼트를 만듭니다. F는 같은 값이 많아(절반이 2회 이하) 순위로 바꾼 뒤 다섯 구간으로 나눕니다.

In [16]:
# [제공] RFM 점수와 규칙 세그먼트
r_score = pd.qcut(rfm["R"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
f_score = pd.qcut(rfm["F"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
conditions = [
    (r_score >= 4) & (f_score >= 4),
    (r_score >= 4),
    (r_score <= 2) & (f_score >= 4),
    (r_score <= 2),
]
rfm["rule"] = np.select(conditions, ["VIP", "최근 신규", "이탈 위험 우량", "휴면"], default="일반")
rfm["rule"].value_counts()

rule
휴면          1364
VIP         1028
일반           853
최근 신규        679
이탈 위험 우량     342
Name: count, dtype: int64

**[직접 작성]** 두 가지를 출력하세요.

1. 규칙 세그먼트별 고객 수, R·F·M 중앙값, 다음 해 재구매율, 다음 해 구매액 평균
2. `pd.crosstab(rfm["rule"], rfm["group"])`으로 규칙 세그먼트와 K-means 그룹 겹쳐 보기

In [17]:
rule_table = rfm.groupby("rule").agg(고객수=("R", "size"), R=("R", "median"), F=("F", "median"), M=("M", "median"),
                                      다음해_재구매율=("repurchase", "mean"), 다음해_구매액_평균=("next_amount", "mean"))
rule_table["다음해_재구매율"] = rule_table["다음해_재구매율"].round(3)
rule_table["다음해_구매액_평균"] = rule_table["다음해_구매액_평균"].round(1)
print(rule_table.sort_values("다음해_재구매율", ascending=False))
print()
print("[규칙 세그먼트 × K-means 그룹]")
pd.crosstab(rfm["rule"], rfm["group"])

           고객수      R    F         M  다음해_재구매율  다음해_구매액_평균
rule                                                      
VIP       1028    9.0  7.0  2234.995     0.912      4579.0
이탈 위험 우량   342  104.5  4.0  1369.575     0.708      1403.1
일반         853   47.0  2.0   746.240     0.666      1232.4
최근 신규      679   14.0  2.0   436.700     0.626       773.0
휴면        1364  190.0  1.0   310.585     0.396       312.6

[규칙 세그먼트 × K-means 그룹]


group,성장 중,최근 1회 구매,핵심 고객,휴면 1회 구매
rule,,,,
VIP,526,18,484,0
이탈 위험 우량,243,9,41,49
일반,369,396,88,0
최근 신규,184,495,0,0
휴면,129,372,0,863


> **[판단]** 규칙 세그먼트도 다음 해 행동을 가르나요? K-means의 "핵심 고객"과 규칙의 "VIP"는 어떻게 다른가요? K-means에는 없는데 규칙에는 있는 세그먼트는 무엇이고, 그 세그먼트는 CRM팀에 왜 중요할까요?
>
> 나의 답:

규칙 세그먼트도 다음 해 행동을 가릅니다. VIP 91.2%(구매액 4,579), 이탈 위험 우량 70.8%, 일반 66.6%, 최근 신규 62.6%, 휴면 39.6%입니다. K-means의 "핵심 고객"(95.4%)과 규칙의 "VIP"(91.2%)는 비슷하지만 다릅니다. 교차표를 보면 VIP 1,028명 중 484명만 핵심 고객이고 **526명은 성장 중**입니다. 규칙 VIP는 "R 4~5점 그리고 F 4~5점"이라 F가 중간인 성장 중 고객도 포함하지만, K-means는 F·M이 모두 높은 쪽을 핵심으로 묶습니다. 즉 **규칙은 더 넓고, K-means는 더 좁고 순수**합니다.
K-means에는 없는데 규칙에는 있는 세그먼트는 **"이탈 위험 우량"(342명, 재구매율 70.8%)** 입니다. 이들은 과거에 자주 사다가(R 104일로 오래됨, F 4회) 최근에 끊긴 고객으로, K-means에서는 성장 중(243명)이나 휴면(49명)·핵심(41명)에 흩어져 사라집니다. CRM팀에는 **매우 중요합니다**. 아직 가치가 있고 되돌아올 가능성이 높은 "떠나가는 고객"이기 때문입니다. K-means는 R·F·M을 동시에 보느라 이 조합을 별도 그룹으로 만들지 못했습니다.


> **[판단]** 의뢰서의 제약 "그룹이 자주 바뀌면 곤란하다"를 생각하면 K-means와 규칙 중 어느 쪽이 유리한가요? 둘을 함께 쓴다면 어떻게 쓰겠습니까?
>
> 나의 답:

**규칙이 유리합니다.** 규칙의 경계(분위 5구간)는 다음 해 데이터가 들어와도 **고정**되어 있어, 새 고객을 넣어도 그룹이 거의 바뀌지 않습니다. K-means는 데이터가 조금만 늘어도 중심이 이동해 **모든 고객의 그룹이 재배정**될 수 있습니다(이번에도 초기값에 따라 ARI 0.997~1.000으로 미세하게 달라졌습니다). "그룹이 자주 바뀌면 곤란하다"는 제약에는 규칙이 맞습니다.
둘을 함께 쓰는 방법을 제안합니다. **운영의 기준은 규칙**(분기마다 같은 분위 기준으로 재계산, 경계는 고정), **규칙 안의 우선순위는 K-means나 모델 확률**(예: VIP 1,028명 중 "핵심 고객" 484명에게 먼저 전담 상담, "성장 중" 526명에게는 리마인드)로 정합니다. 즉 **규칙은 그룹을 안정적으로 유지하고, K-means는 그룹 내부의 차이를 설명**하는 역할입니다. 이렇게 하면 운영팀이 기억할 그룹은 5개로 유지하면서, 정책의 강도를 세밀하게 조절할 수 있습니다.


---
# Part 4. CRM팀에 보고하기

## 4-1. 쓸 수 있는 문장과 없는 문장

다음 문장들이 이 실습의 결과로 **쓸 수 있는 문장인지** 판단하고, 쓸 수 없다면 이유를 적으세요.

1. "고객을 R·F·M 기준으로 4개 그룹으로 나눴다. 각 그룹의 기준값은 표와 같다."
2. "데이터 분석 결과 우리 고객은 본래 4개 유형으로 나뉜다."
3. "네 그룹은 다음 해 재구매율이 35%에서 95%까지 다르다."
4. "초기값을 바꿔도 결과가 같았으므로(ARI 1.0에 가까움) 이 군집은 유일한 정답이다."
5. "이탈 위험 고객은 K-means 결과에 나타나지 않으므로 우리 고객 중에는 없다."

> **[판단]** 1~5번 각각: 쓸 수 있다 / 쓸 수 없다(이유)
>
> 나의 답:

1. **쓸 수 있다.** "R·F·M 기준으로 4개 그룹으로 나눴고 기준값은 표와 같다"는 분석가의 **선택과 방법**을 서술한 문장입니다. 근거(중앙값·고객 수)도 함께 있습니다.
2. **쓸 수 없다 — 발견의 함정.** "본래 4개 유형으로 나뉜다"는 말은 데이터가 4개 그룹을 **품고 있었다**고 주장합니다. 실제로는 우리가 고른 전처리(로그+표준화)와 k=4에 따른 결과이고, 계층적 군집과의 ARI는 0.81이었습니다. "우리가 4개로 나눴다"가 정직합니다.
3. **쓸 수 있다.** "다음 해 재구매율이 35%에서 95%까지 다르다"는 **군집에 쓰지 않은 데이터**로 확인한 사실입니다(34.8%~95.4%). 조건(다음 해, 재구매율)을 밝혔으므로 OK입니다.
4. **쓸 수 없다 — 유일한 정답의 함정.** ARI 1.0은 "같은 방법으로 다시 돌려도 같은 답"이라는 뜻(안정성)일 뿐, "유일한 정답"이라는 뜻이 아닙니다. 방법(계층적 군집)을 바꾸면 ARI 0.81로 달라집니다.
5. **쓸 수 없다 — 부재의 오류.** K-means 결과에 없다는 것은 "그런 고객이 없다"가 아니라 "우리가 고른 4개 그룹에 그런 조합이 별도 그룹으로 나타나지 않았다"는 뜻입니다. 규칙으로는 "이탈 위험 우량" 342명이 확인됩니다. 분석에 없다고 세상에 없는 것이 아닙니다.


## 4-2. CRM팀 보고 메모

분석 결과를 CRM팀장에게 메시지로 보낸다고 생각하고 쓰세요. 팀장은 머신러닝 용어를 모릅니다. 아래 네 가지가 모두 들어가야 합니다.

1. **그룹 수와 각 그룹의 기준**: 이름과 함께 R·F·M 기준 숫자
2. **분석가의 선택**: 어떤 정보로, 어떻게 가공해, 왜 4개로 나눴는지 (다른 선택이 가능했다는 것도)
3. **믿을 만하다는 근거**: 군집에 쓰지 않은 다음 해 행동
4. **제안과 주의할 점**: 그룹별 정책을 어떻게 시험할지, 그룹 구분을 얼마나 자주 다시 볼지, 규칙과 함께 쓰는 방법

> **[판단]** CRM팀 보고 메모:
>
> 1. 그룹 수와 기준:
>
> 2. 분석가의 선택:
>
> 3. 믿을 만하다는 근거:
>
> 4. 제안과 주의할 점:
>
> 나의 답:

**CRM팀 보고 메모 (팀장님께)**
1. **그룹 수와 기준**: 고객 4,266명을 **4개 그룹**으로 나눴습니다. ⑴ **핵심 고객** 613명 — 최근 12일 안에 사고, 1년에 10번, 총 4,215 구매. ⑵ **성장 중** 1,451명 — 최근 34일, 4번, 1,307. ⑶ **최근 1회 구매** 1,290명 — 최근 43일, 1번, 329. ⑷ **휴면 1회 구매** 912명 — 243일 전에 1번, 286. (숫자는 중앙값)
2. **분석가의 선택**: 2024년(2009.12~2010.11) 구매 기록만 쓰고, 취소·반품·고객 ID 없는 주문을 제외한 805,549건(전체의 75.5%)으로 분석했습니다. 금액이 한쪽으로 크게 치우쳐 있어 로그를 취하고 표준화했습니다. 그룹 수는 4개로 정했습니다. 2~8개를 모두 돌려 보니 통계적으로는 3개도 가능했지만, "성장 중"과 "최근 1회 구매"를 한 덩어리로 묶으면 재구매율 79%와 52%가 섞여 정책을 나눌 수 없었습니다. **다른 선택(3개, 다른 전처리)도 가능했다**는 점을 밝혀 둡니다.
3. **믿을 만하다는 근거**: 그룹을 나눌 때 **쓰지 않은** 다음 해(2010.12~2011.11) 행동을 확인했습니다. 재구매율이 **핵심 95.4% / 성장 중 78.5% / 최근 1회 52.1% / 휴면 34.8%** 로 갈렸습니다(전체 63.6%). 특히 "최근 1회"와 "휴면 1회"는 구매 횟수·금액이 거의 같은데 재구매율이 17%p 차이 나, 최근성으로 나눈 것이 실제로 의미 있었습니다. 초기값을 5번 바꿔도 결과가 거의 같았습니다(ARI 0.997~1.000).
4. **제안과 주의할 점**: ⑴ **핵심 고객**은 전담 상담·우선 배송, **성장 중**은 리마인드·번들 제안, **최근 1회**는 2회차 유도 쿠폰, **휴면 1회**는 저비용 이메일 캠페인으로 정책을 나눠 주세요. ⑵ 지금 쓰는 RFM 점수 규칙과 함께 쓰시길 권합니다. 규칙은 경계가 고정되어 그룹이 자주 바뀌지 않고, 규칙에서만 보이는 **"이탈 위험 우량"(342명, 재구매율 70.8%)** 은 K-means에는 나타나지 않습니다(성장·휴면·핵심에 흩어짐). ⑶ 그룹은 **분기마다** 다시 계산하고 경계는 고정하세요. ⑷ 이 그룹은 "고객이 본래 그렇다"가 아니라 "이 기준으로 나눴다"는 것입니다. 다른 방법으로 나누면 달라집니다(계층적 군집과 81%만 일치). ⑸ 가장 중요한 주의: 이 차이는 **관찰된 차이**이고, 그룹별 정책이 실제로 효과가 있는지는 **무작위로 나눠 시험**해야 알 수 있습니다. ⑹ 데이터는 영국 온라인 도매몰 2009~2011년 기록입니다.


---
# 마무리

## 이 Chapter의 판단 포인트

> **군집은 발견이 아니라 선택이다. 전처리와 $k$를 바꾸면 그룹이 바뀐다. 그룹이 쓸모 있는지는 다음 기간의 행동으로 확인한다.**

## 개인 검증 체크리스트에 추가

오늘 실습에서 내가 실제로 겪은 일을 바탕으로, 체크리스트의 줄을 내 말로 적으세요.

예: "군집 결과를 보면 '전처리는 무엇이었고 $k$는 왜 그 값인가'를 먼저 묻는다."

> **[판단]** 나의 체크리스트 10번(군집 결과를 받았을 때):
>
> 나의 답:

군집 결과를 받으면 (1) **전처리가 무엇이었고 k는 왜 그 값인가**를 먼저 묻고, (2) 실루엣 같은 숫자보다 **그룹이 쓸모 있는가**(그룹별로 다른 행동을 하는가)를 확인하며, (3) 그룹을 만드는 데 쓰지 않은 정보로 검증했는지, (4) 방법을 바꾸면 그룹이 얼마나 달라지는지(ARI)를 함께 본다.


> **[판단]** 나의 체크리스트 11번(정답이 없는 결과를 검증할 때):
>
> 나의 답:

정답이 없는 결과를 검증할 때는 (1) **미래 기간**이나 **쓰지 않은 정보**로 그룹 간 차이가 실제로 나타나는지 보고, (2) 그 차이가 **그룹 때문인지 다른 변수 때문인지** 구분하며, (3) 정책의 효과는 관찰이 아니라 **실험**으로 확인한다. "안정적"과 "유일한 답"은 다른 말이다.


## 제출물

- 이 노트북 (모든 **[판단]** 칸을 채운 상태)

수고했습니다. 다음 Chapter 7에서는 차원 축소, 이상 탐지, 장바구니 분석으로 정답 없는 데이터에서 패턴을 찾고, 그 패턴이 내가 고른 기준의 결과인지 확인합니다.